# 09: Reproducibility Audit

Independent re-execution of a fixed random sample of the 960 runs,
reconstructed from the primitive pipeline functions (bypassing the
run_experiment wrapper so no W&B runs are generated) and compared
value-for-value against results_master.csv.

Claim verified: given the archived code, processed data, frozen
environment, and fixed seeds, any run in the study reproduces its
recorded metrics exactly.

Environment rule: install from requirements.txt only (the frozen
execution-time versions); never -U in this notebook.

In [ ]:
%pip install -q -r requirements.txt

In [2]:
import os, re
os.environ["WANDB_MODE"] = "offline"   # belt-and-suspenders for the import

import pandas as pd
import numpy as np
from manipulation_functions import *
from experiment_functions import train_model, compute_metrics

BUCKET = "osilesi-dissertation-data-2026"

def sanitize_columns(df):
    df = df.copy()
    df.columns = [re.sub(r"[\[\]<>]", "_", str(c)) for c in df.columns]
    assert df.columns.is_unique
    return df

datasets = {
    "adult":  sanitize_columns(pd.read_csv(
        f"s3://{BUCKET}/processed/adult_final_pruned.csv")),
    "health": sanitize_columns(pd.read_csv(
        f"s3://{BUCKET}/processed/health_final_pruned.csv")),
}

res = pd.read_csv(f"s3://{BUCKET}/results/results_master.csv")
res["min_level"] = res["min_level"].astype(str)   # LEVELS keys are strings
print(res.shape, "| datasets:",
      {k: v.shape for k, v in datasets.items()})

(960, 19) | datasets: {'adult': (32561, 43), 'health': (101766, 109)}


## Sample construction

Ten runs, fixed audit seed (99) so the audit itself is reproducible,
drawn as a stratified sample guaranteeing coverage of both datasets and
all three classifiers, and explicitly including at least one
health/XGBoost run: those 160 runs failed pre-sanitization, so
reproducing one from the current pipeline is direct evidence the
column-name fix is complete and metric-invariant.

In [3]:
rng = np.random.default_rng(99)

# One run per dataset x classifier stratum (6), plus 4 more at random
strata_picks = (res.groupby(["dataset", "classifier"])
                   .sample(n=1, random_state=99))
remaining = res.drop(strata_picks.index)
extra = remaining.sample(n=4, random_state=99)
sample = pd.concat([strata_picks, extra]).sort_values("run_id")

assert len(sample) == 10
assert ((sample["dataset"] == "health") &
        (sample["classifier"] == "xgboost")).any(), \
    "sample must include a health/xgboost run"
sample[["run_id", "dataset", "classifier", "imbalance",
        "min_level", "min_type", "seed"]]

,run_id,dataset,classifier,imbalance,min_level,min_type,seed
91,91,adult,logistic_regression,90/10,75,horizontal,22
137,137,adult,logistic_regression,95/5,75,vertical,33
282,282,adult,random_forest,95/5,100,horizontal,33
352,352,adult,xgboost,50/50,25,horizontal,33
428,428,adult,xgboost,90/10,50,vertical,44
523,523,health,logistic_regression,80/20,100,horizontal,44
608,608,health,logistic_regression,95/5,100,vertical,44
684,684,health,random_forest,80/20,100,horizontal,55
723,723,health,random_forest,90/10,100,horizontal,44
801,801,health,xgboost,50/50,100,horizontal,22


In [4]:
METRICS = ["accuracy", "minority_recall", "macro_f1",
           "balanced_accuracy", "mcc",
           "equal_opportunity_diff", "fnr_diff"]

audit_rows = []
for _, row in sample.iterrows():
    d = apply_imbalance(datasets[row["dataset"]], row["imbalance"],
                        int(row["seed"]), base_n=15000)
    if row["min_type"] == "horizontal":
        d = apply_horizontal_minimization(d, row["min_level"],
                                          int(row["seed"]))
    else:
        d = apply_vertical_minimization(d, row["min_level"],
                                        int(row["seed"]))
    train, test = make_split(d, int(row["seed"]))
    model = train_model(train, row["classifier"], int(row["seed"]))
    m = compute_metrics(model, test)

    for k in METRICS:
        audit_rows.append({
            "run_id": int(row["run_id"]), "metric": k,
            "original": float(row[k]), "reproduced": float(m[k]),
            "abs_diff": abs(float(row[k]) - float(m[k])),
        })
    print(f"run {int(row['run_id']):>3} "
          f"({row['dataset']}/{row['classifier']}) reproduced")

audit = pd.DataFrame(audit_rows)
print("\nmax absolute difference:", audit["abs_diff"].max())

run  91 (adult/logistic_regression) reproduced
run 137 (adult/logistic_regression) reproduced
run 282 (adult/random_forest) reproduced
run 352 (adult/xgboost) reproduced
run 428 (adult/xgboost) reproduced
run 523 (health/logistic_regression) reproduced
run 608 (health/logistic_regression) reproduced
run 684 (health/random_forest) reproduced
run 723 (health/random_forest) reproduced
run 801 (health/xgboost) reproduced

max absolute difference: 1.1102230246251565e-16


In [5]:
TOL_EXACT = 1e-9      # same-machine, frozen-environment standard
TOL_FLOAT = 1e-12     # floating-point summation noise

exact = (audit["abs_diff"] < TOL_EXACT).all()
assert exact, audit[audit["abs_diff"] >= TOL_EXACT]

audit.to_csv("reproducibility_audit.csv", index=False)
res_bucket = f"s3://{BUCKET}/results/reproducibility_audit.csv"
audit.to_csv(res_bucket, index=False)
print(f"AUDIT PASSED: {len(sample)} runs, {len(audit)} metric values, "
      f"all within {TOL_EXACT}")

AUDIT PASSED: 10 runs, 70 metric values, all within 1e-09
